In [1]:
import pandas as pd, numpy as np
from sklearn.neighbors import NearestNeighbors

CORE = ["tensile_strength_MPa", "yield_strength_MPa", "elongation_pct",
        "youngs_modulus_GPa", "density_g_cm3"]

all_df = pd.read_csv("../data/processed/materials_clustered_all.csv")
all_df["elongation_assumed"] = all_df["elongation_assumed"].fillna(False).astype(bool)
all_df.loc[all_df["elongation_assumed"], "elongation_pct"] = np.nan   # grey iron: "not reported"
all_df["specific_strength"]  = all_df["tensile_strength_MPa"] / all_df["density_g_cm3"]
all_df["specific_stiffness"] = all_df["youngs_modulus_GPa"] / all_df["density_g_cm3"]
print(all_df.shape)
all_df["cluster_name"].value_counts()

(1256, 34)


cluster_name
Medium-strength ductile steels             529
High-strength steels                       336
Light alloys (Al, Mg)                      233
Ductile copper alloys                      137
Grey cast iron (brittle, not clustered)     21
Name: count, dtype: int64

In [2]:
SHOW = ["label", "family", "cluster_name", "tensile_strength_MPa", "yield_strength_MPa",
        "elongation_pct", "youngs_modulus_GPa", "density_g_cm3", "specific_strength"]

def recommend(constraints, rank_by="specific_strength", ascending=False, top=8, families=None):
    """constraints: {column: (min, max)}; use None for no limit."""
    mask = pd.Series(True, index=all_df.index)
    for col, (lo, hi) in constraints.items():
        if lo is not None:
            mask &= all_df[col] >= lo
        if hi is not None:
            mask &= all_df[col] <= hi
    if families:
        mask &= all_df["family"].isin(families)
    cand = all_df[mask]
    if cand.empty:
        return "No material satisfies all constraints. Relax one."
    print(f"{len(cand)} materials match. Clusters: {cand['cluster_name'].value_counts().to_dict()}")
    return cand.sort_values(rank_by, ascending=ascending).head(top)[SHOW].round(1).reset_index(drop=True)

In [3]:
# A) Lightweight structural part
recommend({"density_g_cm3": (None, 3.0), "tensile_strength_MPa": (300, None)})

65 materials match. Clusters: {'Light alloys (Al, Mg)': 65}


,label,family,cluster_name,tensile_strength_MPa,yield_strength_MPa,elongation_pct,youngs_modulus_GPa,density_g_cm3,specific_strength
0,Aluminum Alloy 7178-T651 | Wrought,Aluminium,"Light alloys (Al, Mg)",607.0,538.0,10.0,73.0,2.7,224.8
1,Magnesium Alloy AZ80A-T5 | As extruded,Magnesium,"Light alloys (Al, Mg)",379.0,276.0,7.0,73.0,1.8,216.6
2,Aluminum Alloy 7178-T7651 | Wrought,Aluminium,"Light alloys (Al, Mg)",572.0,503.0,10.0,73.0,2.7,211.9
3,Aluminum Alloy 7075-T6 | Wrought,Aluminium,"Light alloys (Al, Mg)",572.0,503.0,11.0,73.0,2.7,211.9
4,Aluminum Alloy Alclad 7178-T651 | Wrought,Aluminium,"Light alloys (Al, Mg)",558.0,490.0,10.0,73.0,2.7,206.7
5,Aluminum Alloy 7050-T7651 | Wrought,Aluminium,"Light alloys (Al, Mg)",552.0,490.0,11.0,73.0,2.7,204.4
6,Magnesium Alloy ZK60A-F | As extruded,Magnesium,"Light alloys (Al, Mg)",352.0,283.0,11.0,73.0,1.8,201.1
7,Aluminum Alloy Alclad 7075-T651 | Wrought,Aluminium,"Light alloys (Al, Mg)",524.0,462.0,11.0,73.0,2.7,194.1


In [4]:
# B) Highly loaded shaft: very strong, but not brittle
recommend({"tensile_strength_MPa": (900, None), "elongation_pct": (10, None)},
          rank_by="yield_strength_MPa")

99 materials match. Clusters: {'High-strength steels': 99}


,label,family,cluster_name,tensile_strength_MPa,yield_strength_MPa,elongation_pct,youngs_modulus_GPa,density_g_cm3,specific_strength
0,Steel SAE 4150 | tempered at 400 F,Steel,High-strength steels,1931.0,1724.0,10.0,207.0,7.9,245.7
1,Steel SAE 4340 | tempered at 400 F,Steel,High-strength steels,1875.0,1675.0,10.0,207.0,7.9,238.5
2,Steel SAE 8650 | tempered at 400 F,Steel,High-strength steels,1937.0,1675.0,10.0,207.0,7.9,246.4
3,Steel SAE 8640 | tempered at 400 F,Steel,High-strength steels,1862.0,1669.0,10.0,207.0,7.9,236.9
4,Steel SAE 4042 | tempered at 400 F,Steel,High-strength steels,1800.0,1662.0,12.0,207.0,7.9,229.0
5,Steel SAE 8740 | tempered at 400 F,Steel,High-strength steels,1999.0,1655.0,10.0,207.0,7.9,254.3
6,Steel SAE 1340 | tempered at 400 F,Steel,High-strength steels,1806.0,1593.0,11.0,207.0,7.9,229.8
7,Steel SAE 8660 | tempered at 800 F,Steel,High-strength steels,1634.0,1551.0,13.0,207.0,7.9,207.9


In [5]:
# C) Formed or deep-drawn part: very ductile, moderate strength
recommend({"elongation_pct": (30, None), "tensile_strength_MPa": (300, None)},
          rank_by="elongation_pct")

157 materials match. Clusters: {'Medium-strength ductile steels': 99, 'Ductile copper alloys': 58}


,label,family,cluster_name,tensile_strength_MPa,yield_strength_MPa,elongation_pct,youngs_modulus_GPa,density_g_cm3,specific_strength
0,Copper Alloy C52400 | n/a,Copper alloy,Ductile copper alloys,455.0,193.0,70.0,105.0,8.8,51.7
1,Copper Alloy C52100 | n/a,Copper alloy,Ductile copper alloys,379.0,165.0,70.0,105.0,8.8,43.1
2,Cartridge brass C26000 | n/a,Copper alloy,Ductile copper alloys,303.0,76.0,66.0,105.0,8.8,34.4
3,Medium-leaded brass C35000 | n/a,Copper alloy,Ductile copper alloys,310.0,90.0,66.0,105.0,8.8,35.2
4,Yellow brass C26800 | n/a,Copper alloy,Ductile copper alloys,317.0,97.0,65.0,105.0,8.8,36.0
5,Copper Alloy C44300 | n/a,Copper alloy,Ductile copper alloys,331.0,124.0,65.0,105.0,8.8,37.6
6,Copper Alloy C51000 | n/a,Copper alloy,Ductile copper alloys,324.0,131.0,64.0,105.0,8.8,36.8
7,Copper Alloy C65500 | n/a,Copper alloy,Ductile copper alloys,386.0,145.0,63.0,105.0,8.8,43.9


In [6]:
X = pd.read_csv("../data/processed/materials_scaled.csv", index_col=0)
meta = pd.read_csv("../data/processed/materials_clustered.csv", index_col=0).loc[X.index]
nn = NearestNeighbors(n_neighbors=11).fit(X)

def find(text):
    return [l for l in X.index if text.lower() in l.lower()][:15]

def similar_to(label, n=5):
    i = X.index.get_loc(label)
    dist, idx = nn.kneighbors(X.iloc[[i]], n_neighbors=n + 1)
    out = meta.loc[X.index[idx[0][1:]], ["family", "cluster_name"] + CORE].copy()
    out["distance"] = dist[0][1:].round(2)
    return out

find("SAE 4")            # look at the names it returns, then pick one

['Steel SAE 4037 | tempered at 400 F',
 'Steel SAE 4042 | tempered at 400 F',
 'Steel SAE 4130 | normalized',
 'Steel SAE 4130 | annealed',
 'Steel SAE 4130 | tempered at 400 F',
 'Steel SAE 4140 | normalized',
 'Steel SAE 4140 | annealed',
 'Steel SAE 4140 | tempered at 400 F',
 'Steel SAE 4150 | normalized',
 'Steel SAE 4150 | annealed',
 'Steel SAE 4150 | tempered at 400 F',
 'Steel SAE 4320 | normalized',
 'Steel SAE 4320 | annealed',
 'Steel SAE 4340 | normalized',
 'Steel SAE 4340 | annealed']

In [7]:
def p10(s): return s.quantile(0.1)
def p90(s): return s.quantile(0.9)
datasheet = meta.groupby("cluster_name")[CORE].agg([p10, "median", p90]).round(1)
datasheet

tensile_strength_MPa                 \
                                                p10 median     p90   
cluster_name                                                         
Ductile copper alloys                         228.8  310.0   469.0   
High-strength steels                          686.0  925.0  1500.0   
Light alloys (Al, Mg)                         131.0  241.0   433.2   
Medium-strength ductile steels                400.0  540.0   700.0   

                               yield_strength_MPa                 \
                                              p10 median     p90   
cluster_name                                                       
Ductile copper alloys                        80.2  131.0   223.8   
High-strength steels                        462.0  685.0  1300.0   
Light alloys (Al, Mg)                        77.4  172.0   345.0   
Medium-strength ductile steels              206.6  315.0   430.0   

                               elongation_pct              youngs_modulus_GPa  \
                                          p10 median   p90                p10   
cluster_name                                                                    
Ductile copper alloys                    19.6   35.0  55.0              100.0   
High-strength steels                      5.0   11.0  14.0              199.0   
Light alloys (Al, Mg)                     3.0   10.0  22.0               73.0   
Medium-strength ductile steels           12.6   20.0  40.0              193.0   

                                             density_g_cm3              
                               median    p90           p10 median  p90  
cluster_name                                                            
Ductile copper alloys           105.0  120.0           8.5    8.8  8.9  
High-strength steels            206.0  213.0           7.8    7.9  7.9  
Light alloys (Al, Mg)            73.0   73.0           1.8    2.7  2.7  
Medium-strength ductile steels  206.0  207.0           7.8    7.9  7.9

**Scope.** This is a first-pass screening tool for structural metals (steel, stainless, aluminium, copper alloys, cast iron, magnesium) from the Autodesk Material Library data. It uses room-temperature strength, yield, ductility, stiffness and density only. It does not account for creep, fatigue, oxidation/corrosion, maximum service temperature, thermal conductivity or cost, all of which matter for turbine and engine parts. Shortlists must be checked against those before selection.

In [8]:
similar_to("Steel SAE 4140 | normalized")

,family,cluster_name,tensile_strength_MPa,yield_strength_MPa,elongation_pct,youngs_modulus_GPa,density_g_cm3,distance
label,,,,,,,,
Steel SAE 8740 | normalized,Steel,High-strength steels,929.0,607.0,16.0,207.0,7.86,0.24
Steel SAE 9255 | normalized,Steel,High-strength steels,933.0,579.0,19.7,207.0,7.86,0.27
Steel SAE 1144 | tempered at 400 F,Steel,High-strength steels,876.0,627.0,17.0,207.0,7.86,0.28
JIS SUH1 | n/a,Steel,High-strength steels,930.0,685.0,15.0,206.0,7.86,0.28
Steel SAE 9310 | normalized,Steel,High-strength steels,907.0,571.0,18.8,207.0,7.86,0.29


In [9]:
similar_to("Cartridge brass C26000 | n/a")

,family,cluster_name,tensile_strength_MPa,yield_strength_MPa,elongation_pct,youngs_modulus_GPa,density_g_cm3,distance
label,,,,,,,,
Copper Alloy C66700 | n/a,Copper alloy,Ductile copper alloys,316.0,83.0,60.0,105.0,8.8,0.19
Medium-leaded brass C35000 | n/a,Copper alloy,Ductile copper alloys,310.0,90.0,66.0,105.0,8.8,0.24
Low brass C24000 | n/a,Copper alloy,Ductile copper alloys,290.0,83.0,55.0,105.0,8.8,0.29
Red brass C23000 | n/a,Copper alloy,Ductile copper alloys,269.0,69.0,55.0,105.0,8.8,0.35
Yellow brass C26800 | n/a,Copper alloy,Ductile copper alloys,317.0,97.0,65.0,105.0,8.8,0.35


In [10]:
similar_to("Aluminum Alloy 7075-T6 | Wrought")

,family,cluster_name,tensile_strength_MPa,yield_strength_MPa,elongation_pct,youngs_modulus_GPa,density_g_cm3,distance
label,,,,,,,,
Aluminum Alloy 7050-T7651 | Wrought,Aluminium,"Light alloys (Al, Mg)",552.0,490.0,11.0,73.0,2.7,0.07
Aluminum Alloy 7178-T7651 | Wrought,Aluminium,"Light alloys (Al, Mg)",572.0,503.0,10.0,73.0,2.7,0.13
Aluminum Alloy Alclad 7178-T651 | Wrought,Aluminium,"Light alloys (Al, Mg)",558.0,490.0,10.0,73.0,2.7,0.14
Aluminum Alloy 7050-T7451f | Wrought,Aluminium,"Light alloys (Al, Mg)",524.0,469.0,11.0,73.0,2.7,0.18
Aluminum Alloy 7178-T651 | Wrought,Aluminium,"Light alloys (Al, Mg)",607.0,538.0,10.0,73.0,2.7,0.19


In [11]:
similar_to("BS CR4GP | n/a")      # if not found, use find("CR4") to get the exact label

,family,cluster_name,tensile_strength_MPa,yield_strength_MPa,elongation_pct,youngs_modulus_GPa,density_g_cm3,distance
label,,,,,,,,
BS CR3GP | n/a,Steel,Ductile copper alloys,280.0,140.0,32.0,206.0,7.860,0.07
BS CR2PL | n/a,Steel,Ductile copper alloys,280.0,140.0,34.0,206.0,7.860,0.15
NF A34-2 | n/a,Steel,Medium-strength ductile steels,330.0,155.0,27.0,206.0,7.860,0.36
NF XE | n/a,Steel,Medium-strength ductile steels,270.0,180.0,34.0,206.0,7.860,0.39
Steel 10 GOST 1050-88 | Annealing or high tempering,Steel,Medium-strength ductile steels,290.0,190.0,26.0,206.0,7.856,0.48


**Scenario results.** A (density ≤ 3, tensile ≥ 300 MPa) returns only light alloys, led by 7xxx aluminium (7178, 7075, 7050) and magnesium AZ80A/ZK60A, ranked by specific strength (up to ~225 MPa per g/cm³). B (tensile ≥ 900 MPa, elongation ≥ 10%) returns only high-strength steels (4150, 4340, 8650 tempered at 400°F; ~1,800-2,000 MPa, ~10% elongation), typical shaft materials. C (elongation ≥ 30%, tensile ≥ 300 MPa) returns phosphor bronzes and brasses first (60-70% elongation), with medium-strength steels also among the 157 matches. In each case the constraints select one or two clusters, so the clusters correspond to groups an engineer would ask for.